In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


In [5]:
# 1. Set project paths


try:
    PROJECT_DIR = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_DIR = Path.cwd().parent


INPUT_FILE = PROJECT_DIR / "data" / "processed" / "clean_stocks_data.csv"
OUTPUT_DIR = PROJECT_DIR / "reports"
FIGURES_DIR = PROJECT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
# 2. Load and validate the cleaned data

df = pd.read_csv(INPUT_FILE)

required_columns = {"Date", "Ticker", "Close"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Close"] = pd.to_numeric(df["Close"], errors="coerce")

df = df.dropna(subset=["Date", "Ticker", "Close"])
df = df[df["Close"] > 0]

df["Ticker"] = df["Ticker"].astype(str).str.upper().str.strip()
df = df.sort_values(["Ticker", "Date"])

# Each stock should have only one row per trading date.
duplicates = df.duplicated(subset=["Ticker", "Date"])

if duplicates.any():
    raise ValueError(
        "Duplicate ticker/date rows found. Please check the cleaned data."
    )


In [7]:
# 3. Calculate moving averages

windows = [20, 50, 200]

for window in windows:
    column_name = f"MA{window}"
    df[column_name] = (
        df.groupby("Ticker")["Close"]
        .transform(
            lambda prices: prices.rolling(
                window=window,
                min_periods=window
            ).mean()
        )
    )

# Save all prices and moving averages.
output_csv = OUTPUT_DIR / "moving_average_data.csv"
df.to_csv(output_csv, index=False)



In [8]:
# 4. Create a separate chart for each stock

stocks = ["AAPL", "MSFT", "NVDA"]

for ticker in stocks:
    stock = df[df["Ticker"] == ticker].copy()

    if stock.empty:
        print(f"Warning: No data found for {ticker}. Skipping.")
        continue

    fig, ax = plt.subplots(figsize=(13, 7))

    ax.plot(
        stock["Date"],
        stock["Close"],
        label="Adjusted closing price",
        color="#1F2937",
        linewidth=1.5,
        alpha=0.85,
    )

    ax.plot(
        stock["Date"],
        stock["MA20"],
        label="20-day MA",
        linewidth=1.5,
    )

    ax.plot(
        stock["Date"],
        stock["MA50"],
        label="50-day MA",
        linewidth=1.8,
    )

    ax.plot(
        stock["Date"],
        stock["MA200"],
        label="200-day MA",
        linewidth=2.2,
    )

    ax.set_title(
        f"{ticker}: Stock Price and Moving Averages",
        fontsize=16,
        fontweight="bold",
        pad=16,
    )
    ax.set_xlabel("Date", fontsize=11)
    ax.set_ylabel("Adjusted price (USD)", fontsize=11)

    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False, ncol=2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.autofmt_xdate()
    fig.tight_layout()

    output_chart = FIGURES_DIR / f"{ticker.lower()}_moving_averages.png"
    fig.savefig(output_chart, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Chart saved: {output_chart}")




Chart saved: /Users/golden_horde/Desktop/FA_projects/stock-market-analysis/figures/aapl_moving_averages.png
Chart saved: /Users/golden_horde/Desktop/FA_projects/stock-market-analysis/figures/msft_moving_averages.png
Chart saved: /Users/golden_horde/Desktop/FA_projects/stock-market-analysis/figures/nvda_moving_averages.png


In [9]:
# 5. Create a latest moving-average summary

summary_rows = []

for ticker, stock in df.groupby("Ticker"):
    stock = stock.sort_values("Date")
    latest = stock.iloc[-1]

    row = {
        "Ticker": ticker,
        "Latest_Date": latest["Date"].date(),
        "Latest_Adjusted_Close": latest["Close"],
    }

    for window in windows:
        ma_value = latest[f"MA{window}"]
        row[f"MA{window}"] = ma_value

        if pd.notna(ma_value):
            row[f"Price_Above_MA{window}"] = bool(
                latest["Close"] > ma_value
            )
        else:
            row[f"Price_Above_MA{window}"] = pd.NA

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)

summary_file = OUTPUT_DIR / "moving_average_summary.csv"
summary.to_csv(summary_file, index=False)

print("\nMoving-average analysis completed.")
print(f"Detailed data: {output_csv}")
print(f"Summary: {summary_file}")
print("\nLatest moving-average summary:")
print(summary.round(2).to_string(index=False))



Moving-average analysis completed.
Detailed data: /Users/golden_horde/Desktop/FA_projects/stock-market-analysis/reports/moving_average_data.csv
Summary: /Users/golden_horde/Desktop/FA_projects/stock-market-analysis/reports/moving_average_summary.csv

Latest moving-average summary:
Ticker Latest_Date  Latest_Adjusted_Close   MA20  Price_Above_MA20   MA50  Price_Above_MA50  MA200  Price_Above_MA200
  AAPL  2026-10-21                 343.99 337.96              True 327.70              True 293.23               True
  MSFT  2026-10-21                 532.95 525.13              True 507.19              True 434.92               True
  NVDA  2026-10-21                 221.97 226.87             False 222.91             False 203.12               True
